In [ ]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

import argparse, inspect, json, os, pickle, socket, subprocess, warnings, random, math, librosa, shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import lightning as L
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

from functools import reduce
import umap
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import commons, models, utils, losses, lightning_wrapper
from cough_datasets import CoughDatasets, CoughDatasetsCollate, CoughDiseaseBinaryBatchSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier


from sklearn.metrics import (
            confusion_matrix, accuracy_score,
            balanced_accuracy_score,
            roc_auc_score, roc_curve
        )

torch.set_float32_matmul_precision("medium")
cmap = cm.get_cmap("viridis")

In [ ]:
now_experiment = "resnet34_gtgram_deltadelta"
parser = argparse.ArgumentParser()
parser.add_argument("--init", action="store_true")
parser.add_argument("--model_name", type=str, default="try_wavlmlora_downstream")
parser.add_argument("--config_path", type=str, default="configs/ssl_finetuning.json")
args = parser.parse_args(["--model_name", now_experiment])

model_dir = os.path.join("./logs", args.model_name)
config_save_path = os.path.join(model_dir, "config.json")
with open(config_save_path, "r") as f:
    data = f.read()
config = json.loads(data)

hps = utils.HParams(**config)
hps.model_dir = model_dir
hps.data.mae_training = hps.train.mae_training
hps.data.ssccl_training = hps.train.ssccl_training
hps.model.spk_dim = 0

logger = utils.get_logger(hps.model_dir)
import sys, importlib.util, shutil, tempfile
temp_path = tempfile.NamedTemporaryFile(suffix=".py", delete=False).name
shutil.copy(f"{model_dir}/model_net.py.bak", temp_path)
spec = importlib.util.spec_from_file_location("model_net", temp_path)
model_net = importlib.util.module_from_spec(spec)
sys.modules["model_net"] = model_net
spec.loader.exec_module(model_net)
pool_net = getattr(model_net, hps.model.pooling_model)

hps.model.lora_finetune = False
pool_model = pool_net(hps.model.feature_dim, **hps.model)

runner_lightning = lightning_wrapper.CoughClassificationRunner(pool_model, hps=hps, custom_logger=logger, class_weights=[])
runner_lightning = lightning_wrapper.CoughClassificationRunner.load_from_checkpoint(
    os.path.join(f"{hps.model_dir}/best_model.ckpt"),
    model=pool_model,
    hps=hps, custom_logger=logger
)
runner_lightning.eval()

df_train = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.train')
df_test = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.test')

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

df_train = df_train[hps.data.column_order]
df_test = df_test[hps.data.column_order]

collate_fn = CoughDatasetsCollate(hps.data.many_class)
target_labels = df_train[hps.data.target_column]

with open(os.path.join(hps.model_dir, "info_fold.pkl"), "rb") as f:
    info_fold = pickle.load(f)
    best_fold_idx = info_fold["best_fold_idx"]
    fold_metrics = info_fold["fold_metrics"]


val_dataset = CoughDatasets(df_test.values, hps.data,
                                wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle", train=False)
val_loader = DataLoader(val_dataset, num_workers=28, shuffle=False,
                        batch_size=hps.train.batch_size, pin_memory=True, drop_last=False, collate_fn=collate_fn)

test_wavnames = []
test_preds = []
with torch.no_grad():
    for idx, batch in tqdm(enumerate(val_loader), total=len(val_loader)):
        wavnames, audio, _, attention_masks, dse_ids, [patient_ids, _, _] = batch
        audio = audio.cuda()
        attention_masks = attention_masks.cuda()
        out_model = runner_lightning.model.forward(audio, attention_mask=attention_masks)
        logits = out_model['disease_logits']

        probs = torch.sigmoid(logits).squeeze(-1) 
        #preds = (probs >= 0.5).long()

        test_wavnames.extend(wavnames)
        test_preds.append(probs.cpu())

del audio, attention_masks
test_wavnames = np.array(test_wavnames)
test_preds = torch.cat(test_preds, dim=0).numpy()

df_cough_result = pd.DataFrame({
    "path_file": test_wavnames,
    "cough_probs": test_preds
})

In [ ]:
df_test = pd.read_csv(f'/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/{hps.data.metadata_csv}.test')
df_test = df_test.reset_index(drop=True)

df_testcoughs = df_test.merge(
    df_cough_result[['path_file', 'cough_probs']],
    left_on='path_file',
    right_on='path_file',
    how='left'
)[["path_file", "participant", 'cough_probs', "disease_status"]]

# df_testcoughs = (
#     df_testcoughs
#     .groupby("participant", as_index=False)
#     .agg({
#         "path_file": "first",
#         "disease_status": "first",
#         "cough_probs": "mean"
#     })
# )


In [ ]:
df_train = pd.read_csv(f'data/metadata.csv.train')
df_test = pd.read_csv(f'data/metadata.csv.test')

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# valid_participants = set(df_coughtest['participant'].unique())
# df_test = df_test[df_test['participant'].isin(valid_participants)]

df_train = df_train.groupby("participant", as_index=False).nth(0).reset_index(drop=True)
#df_test = df_test.groupby("participant", as_index=False).nth(0).reset_index(drop=True)

In [ ]:
features = ['weight_loss', 'hemoptysis', 'smoker', 'night_sweats']
X_train = df_train[features].astype(float)
X_test = df_test[features].astype(float)

y_train = df_train["disease_status"].astype(int)
y_test = df_test["disease_status"].astype(int)

models = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=1000
        ))
    ]),
    "SVM-RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced"
        ))
    ]),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        max_depth=6,
        class_weight="balanced",
        random_state=42
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}


In [ ]:
models['LogisticRegression'].fit(X_train, y_train)
y_pred = models['LogisticRegression'].predict(X_test)
y_prob = models['LogisticRegression'].predict_proba(X_test)[:, 1]

In [ ]:
df_test['symp_prob'] = y_prob
df_test = df_test[['path_file', 'participant','symp_prob']]

In [ ]:
df_combine2modal = df_test.merge(
    df_testcoughs[['path_file', 'cough_probs', 'disease_status']],
    left_on='path_file',
    right_on='path_file',
    how='left'
)#[['path_file', 'participant', 'disease_status', 'cough_probs', 'symp_prob']]


In [ ]:
# binary labels from unimodal probabilities
df_combine2modal["cough_label"] = (df_combine2modal["cough_probs"] > 0.5).astype(int)
df_combine2modal["sympt_label"] = (df_combine2modal["symp_prob"] > 0.5).astype(int)

# dual-modal probability and label
df_combine2modal["dualmodal_prob"] = df_combine2modal[["cough_probs", "symp_prob"]].mean(axis=1)
df_combine2modal["dualmodal_label"] = (df_combine2modal["dualmodal_prob"] > 0.5).astype(int)

In [ ]:
# accuracy computation
acc_cough = accuracy_score(df_combine2modal["disease_status"], df_combine2modal["cough_label"])
acc_sympt = accuracy_score(df_combine2modal["disease_status"], df_combine2modal["sympt_label"])
acc_dual  = accuracy_score(df_combine2modal["disease_status"], df_combine2modal["dualmodal_label"])

auc_cough = roc_auc_score(df_combine2modal["disease_status"], df_combine2modal["cough_probs"])
auc_sympt = roc_auc_score(df_combine2modal["disease_status"], df_combine2modal["symp_prob"])
auc_dual  = roc_auc_score(df_combine2modal["disease_status"], df_combine2modal["dualmodal_prob"])

results = {
    "cough_accuracy": acc_cough,
    "symptom_accuracy": acc_sympt,
    "dualmodal_accuracy": acc_dual,
    "cough_aucroc": auc_cough,
    "symptom_aucroc": auc_sympt,
    "dualmodal_aucroc": auc_dual,
}

results

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values(
    by="roc_auc", ascending=False
)

print(results_df)